In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
def downcast(df):
    """Reduce el uso de memoria optimizando dtypes."""
    start = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == "object":
            # No tocamos strings acá; las convertiremos a category después si conviene
            continue
        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
    end = df.memory_usage(deep=True).sum() / 1024**2
    print(f"  Memoria: {start:,.1f} MB → {end:,.1f} MB ({100*(1-end/start):.1f}% reducción)")
    return df

In [3]:
files = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
    "installments_payments.csv",
    "HomeCredit_columns_description.csv",
    "sample_submission.csv",
]

audit = []
for f in files:
    path = RAW / f
    if not path.exists():
        print(f"⚠ No encontrado: {f}")
        continue
    print(f"\n=== {f} ===")
    # Para el diccionario y el submission usamos encoding distinto si hace falta
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
    n_rows, n_cols = df.shape
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    n_dup = df.duplicated().sum()
    n_nulls_total = df.isna().sum().sum()
    pct_nulls = 100 * n_nulls_total / (n_rows * n_cols) if n_cols else 0
    print(f"  Filas: {n_rows:,} | Columnas: {n_cols} | Memoria: {mem_mb:,.1f} MB")
    print(f"  Duplicados exactos: {n_dup:,} | % celdas nulas: {pct_nulls:.2f}%")
    audit.append({
        "archivo": f,
        "filas": n_rows,
        "columnas": n_cols,
        "memoria_mb": round(mem_mb, 1),
        "duplicados": n_dup,
        "pct_nulos": round(pct_nulls, 2),
    })
    del df
    gc.collect()

audit_df = pd.DataFrame(audit)
audit_df.to_csv("reports/reports_audit_summary.csv", index=False)
audit_df


=== application_train.csv ===
  Filas: 307,511 | Columnas: 122 | Memoria: 505.0 MB
  Duplicados exactos: 0 | % celdas nulas: 24.40%

=== application_test.csv ===
  Filas: 48,744 | Columnas: 121 | Memoria: 79.7 MB
  Duplicados exactos: 0 | % celdas nulas: 23.81%

=== bureau.csv ===
  Filas: 1,716,428 | Columnas: 17 | Memoria: 472.8 MB
  Duplicados exactos: 0 | % celdas nulas: 13.50%

=== bureau_balance.csv ===
  Filas: 27,299,925 | Columnas: 3 | Memoria: 1,718.3 MB
  Duplicados exactos: 0 | % celdas nulas: 0.00%

=== previous_application.csv ===
  Filas: 1,670,214 | Columnas: 37 | Memoria: 1,703.0 MB
  Duplicados exactos: 0 | % celdas nulas: 17.98%

=== POS_CASH_balance.csv ===
  Filas: 10,001,358 | Columnas: 8 | Memoria: 1,060.9 MB
  Duplicados exactos: 0 | % celdas nulas: 0.07%

=== credit_card_balance.csv ===
  Filas: 3,840,312 | Columnas: 23 | Memoria: 846.4 MB
  Duplicados exactos: 0 | % celdas nulas: 6.65%

=== installments_payments.csv ===
  Filas: 13,605,401 | Columnas: 8 | Mem

,archivo,filas,columnas,memoria_mb,duplicados,pct_nulos
0,application_train.csv,307511,122,505.0,0,24.40
1,application_test.csv,48744,121,79.7,0,23.81
2,bureau.csv,1716428,17,472.8,0,13.50
3,bureau_balance.csv,27299925,3,1718.3,0,0.00
4,previous_application.csv,1670214,37,1703.0,0,17.98
5,POS_CASH_balance.csv,10001358,8,1060.9,0,0.07
6,credit_card_balance.csv,3840312,23,846.4,0,6.65
7,installments_payments.csv,13605401,8,830.4,0,0.01
8,HomeCredit_columns_description.csv,219,5,0.1,0,12.15
9,sample_submission.csv,48744,2,0.7,0,0.00


In [4]:
# Esto es opcional pero MUY recomendable: una vez convertidos, en adelante
# leerás parquets en segundos en vez de minutos.
to_convert = [f for f in files if f.endswith(".csv") and f not in
              ("HomeCredit_columns_description.csv", "sample_submission.csv")]

for f in to_convert:
    print(f"\n→ {f}")
    df = pd.read_csv(RAW / f)
    df = downcast(df)
    out = PROCESSED / f.replace(".csv", ".parquet")
    df.to_parquet(out, index=False)
    print(f"  Guardado: {out.name}")
    del df
    gc.collect()


→ application_train.csv
  Memoria: 505.0 MB → 348.1 MB (31.1% reducción)
  Guardado: application_train.parquet

→ application_test.csv
  Memoria: 79.7 MB → 55.1 MB (30.8% reducción)
  Guardado: application_test.parquet

→ bureau.csv
  Memoria: 472.8 MB → 409.0 MB (13.5% reducción)
  Guardado: bureau.parquet

→ bureau_balance.csv
  Memoria: 1,718.3 MB → 1,431.9 MB (16.7% reducción)
  Guardado: bureau_balance.parquet

→ previous_application.csv
  Memoria: 1,703.0 MB → 1,588.3 MB (6.7% reducción)
  Guardado: previous_application.parquet

→ POS_CASH_balance.csv
  Memoria: 1,060.9 MB → 727.1 MB (31.5% reducción)
  Guardado: POS_CASH_balance.parquet

→ credit_card_balance.csv
  Memoria: 846.4 MB → 652.3 MB (22.9% reducción)
  Guardado: credit_card_balance.parquet

→ installments_payments.csv
  Memoria: 830.4 MB → 493.1 MB (40.6% reducción)
  Guardado: installments_payments.parquet


In [5]:
import pyarrow.parquet as pq

for f in to_convert:
    name = f.replace(".csv", "")
    df = pd.read_parquet(PROCESSED / f"{name}.parquet")
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape}")
    print("\nDtypes:")
    print(df.dtypes.value_counts())
    print("\nHead:")
    print(df.head(3))
    del df
    gc.collect()


=== application_train ===
Shape: (307511, 122)

Dtypes:
float32    64
int8       37
object     16
int32       2
int16       2
float64     1
Name: count, dtype: int64

Head:
   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE NAME_TYPE_SUITE NAME_INCOME_TYPE  \
0      100002       1         Cash loans           M            N               Y             0          202500.0    406597.5      24700.5         351000.0   Unaccompanied          Working   
1      100003       0         Cash loans           F            N               N             0          270000.0   1293502.5      35698.5        1129500.0          Family    State servant   
2      100004       0    Revolving loans           M            Y               Y             0           67500.0    135000.0       6750.0         135000.0   Unaccompanied          Working   

             NAME_EDUCATION_TYPE    NAME_FAMILY_STATUS  N